# 06 — Social Media Charts (Pillow)
Publication-ready PNGs using the @unwelcomedata brand palette, exported to `twitter_landscape` (1600×900) with watermark, via `shared/chart_factory.py`.

Publication charts, rebuilt from the `04-viz` exploration:
1. **Series points split** — 100% stacked bar per series, segments by finishing position
2. **Top 20 performances** — best normalized runs across all 21 series, colored by winner vs non-winner
3. **Runaways vs nail-biters** — winner’s lead over the runner-up
4. **Every contestant, ranked** — the full 105-contestant ranking, exported oversized (1600×2600); NOT for in-feed social cards (see that section for where it can/can't go)

**Reminder caveat (baked into subtitles/sources):** raw totals aren't comparable across series (episode counts differ), so every chart uses *share of series points*. Points are a subjective, comedic score.

In [ ]:
# ===================================================================
# SETUP
# ===================================================================
import sys, os
from pathlib import Path
import duckdb, yaml

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / 'src'))
sys.path.insert(0, str(PROJECT.parent.parent / 'shared'))

from chart_factory import render_chart
from chart_templates import single_ranked_bars  # for custom (non-preset) sizes
from IPython.display import Image as IPImage, display

with open(PROJECT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

conn = duckdb.connect(str(PROJECT / 'data' / 'project.duckdb'))
social_dir = PROJECT / 'outputs' / 'social'
social_dir.mkdir(parents=True, exist_ok=True)

print('✓ chart_factory loaded (Pillow backend)')
print('✓ DuckDB connected')
print(f'✓ Charts export to: {social_dir}')

## Brand palette
Five-step ramp (best → worst) from the @unwelcomedata brand colors, reused across charts.

In [ ]:
C_NAVY   = '#003049'   # winner / best
C_TEAL   = '#005F73'
C_CYAN   = '#0A9396'
C_GOLD   = '#EE9B00'
C_RED    = '#AE2012'   # last / worst
C_ACCENT = '#AE2012'   # highlight
print('✓ Palette loaded')

## Build `chart_` tables in DuckDB
The factory reads `chart_`-prefixed tables. We derive them from `contestant_metrics` (built in `03-prepare`).

In [ ]:
import pandas as pd

df = conn.execute('SELECT * FROM contestant_metrics').df()

# 1. chart_series_share — one row per series, share (%) by finishing POSITION 1..5.
#    (Position order breaks ties deterministically: rank, then points, then name.)
d = df.sort_values(['series', 'series_rank', 'total_points', 'contestant'],
                   ascending=[True, True, False, True]).copy()
d['pos'] = d.groupby('series').cumcount() + 1
series_share = (d.pivot(index='series', columns='pos', values='pct_of_series_points') * 100)
series_share.columns = [f'pos{c}_pct' for c in series_share.columns]
# Contestant name per position, to label segments inline (name + %).
names = d.pivot(index='series', columns='pos', values='contestant')
names.columns = [f'pos{c}_name' for c in names.columns]
series_share = series_share.join(names).reset_index()
series_share['series_label'] = series_share['series'].apply(lambda s: f'Series {s}')
conn.execute('CREATE OR REPLACE TABLE chart_series_share AS SELECT * FROM series_share')

# 2. chart_top_performances — top 20 contestants by share_vs_equal (apples-to-apples).
#    Colored by whether they WON their series (blue) or not (red) — so the chart
#    shows non-winners who out-performed other series' champions.
top = df.sort_values('share_vs_equal', ascending=False).head(20).copy()
top['label'] = top['contestant'] + ' (S' + top['series'].astype(str) + ')'
top['val'] = top['share_vs_equal']
top['val_label'] = top['share_vs_equal'].map(lambda x: f'{x:.2f}x')
top['bar_color'] = top['is_winner'].map(lambda w: C_TEAL if w else C_RED)
conn.execute('CREATE OR REPLACE TABLE chart_top_performances AS SELECT * FROM top')

# 3. chart_winners — the 21 series winners, RANKED by share of series points (%).
#    single_ranked_bars renders rows in order, so sort descending (most dominant on top).
win = df[df['series_rank'] == 1].sort_values('pct_of_series_points', ascending=False).copy()
win['label'] = win['contestant'] + ' (S' + win['series'].astype(str) + ')'
win['val'] = win['pct_of_series_points'] * 100
win['val_label'] = win['val'].map(lambda x: f'{x:.0f}%')
conn.execute('CREATE OR REPLACE TABLE chart_winners AS SELECT * FROM win')

# 4. chart_margins — winner minus runner-up, in share-of-points percentage points.
# Use RAW scoreboard totals here: S20 was a genuine three-way tie at 151 and
# Maisie Adam won a live tie-breaker. Elsewhere she is recorded at 152 (sole
# winner), but for a 'how close was the finish' chart that +1 would hide the
# tightest race in the show's history, so we undo it -> S20 margin = 0.0.
raw = df.copy()
TIE_BREAK_WINNERS = {20}  # series decided by a tie-breaker, not on points
raw.loc[raw['is_winner'] & raw['series'].isin(TIE_BREAK_WINNERS), 'total_points'] -= 1
raw['raw_share'] = raw['total_points'] / raw.groupby('series')['total_points'].transform('sum')
rows = []
for s, grp in raw.groupby('series'):
    g = grp.sort_values('total_points', ascending=False).reset_index(drop=True)
    # Runner-up = the 2nd-highest scorer by position; a tie at the top -> margin 0.
    win_row = g.iloc[0]
    runnerup_share = g.iloc[1]['raw_share']
    rows.append({'series': s, 'winner': win_row['contestant'],
                 'val': (win_row['raw_share'] - runnerup_share) * 100,
                 'tie_break': s in TIE_BREAK_WINNERS})
mar = pd.DataFrame(rows)
mar['label'] = mar['winner'] + ' (S' + mar['series'].astype(str) + ')'
mar['val_label'] = mar['val'].map(lambda x: f'{x:.1f} pts')
mar.loc[mar['tie_break'], 'val_label'] = mar.loc[mar['tie_break'], 'val'].map(
    lambda x: f'{x:.1f} pts · won on tie-break')
# RANKED: biggest runaways on top, closest nail-biters at the bottom.
mar = mar.sort_values('val', ascending=False)
conn.execute('CREATE OR REPLACE TABLE chart_margins AS SELECT * FROM mar')

# 5. chart_all_ranked — ALL 105 contestants by share_vs_equal, winner/non-winner colored.
#    Feeds the oversized 'full ranking' chart (too tall for a social preset).
allr = df.sort_values('share_vs_equal', ascending=False).copy()
allr['label'] = allr['contestant'] + ' (S' + allr['series'].astype(str) + ')'
allr['val'] = allr['share_vs_equal']
allr['val_label'] = allr['share_vs_equal'].map(lambda x: f'{x:.2f}x')
allr['bar_color'] = allr['is_winner'].map(lambda w: C_TEAL if w else C_RED)
conn.execute('CREATE OR REPLACE TABLE chart_all_ranked AS SELECT * FROM allr')

print('✓ Built chart_ tables:',
      [r[0] for r in conn.execute("SHOW TABLES").fetchall() if r[0].startswith('chart_')])

---
## Chart 1: How each series split its points
100% stacked bar per series; segments are finishers by rank (Winner → last). Bars are all 100% wide, so a 5-episode series and a 10-episode series are directly comparable.

In [ ]:
render_chart({
    'type': 'stacked_100pct',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',
    'table': 'chart_series_share',
    'group_col': 'series_label',
    'segments': [
        {'col': 'pos1_pct', 'label': 'Winner', 'color': C_NAVY},
        {'col': 'pos2_pct', 'label': '2nd',    'color': C_TEAL},
        {'col': 'pos3_pct', 'label': '3rd',    'color': C_CYAN},
        {'col': 'pos4_pct', 'label': '4th',    'color': C_GOLD},
        {'col': 'pos5_pct', 'label': 'Last',   'color': C_RED},
    ],
    'segment_label_cols': ['pos1_name', 'pos2_name', 'pos3_name', 'pos4_name', 'pos5_name'],
    'show_legend': False,
    'bar_height': 27,
    'bar_gap': 7,
    'title': 'How each Taskmaster series split its points',
    'subtitle': 'Each bar is one series (100%); segments are contestants by share of series points',
    'source': 'Taskmaster UK series 1–21, contestant point totals',
    'filename': '01_series_points_share',
})

---
## Chart 2: Top 20 performances of all time
Ranked by `share_vs_equal` — share of series points vs an even 1/5 split (1.0 = an average contestant that series). This is the apples-to-apples cross-series ranking; showing the top 20 keeps it legible for social.

In [ ]:
render_chart({
    'type': 'single_ranked_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',
    'table': 'chart_top_performances',
    'category_col': 'label',
    'value_col': 'val',
    'label_col': 'val_label',
    'color_col': 'bar_color',
    'legend': [
        {'label': 'Won their series', 'color': C_TEAL},
        {'label': 'Did not win', 'color': C_RED},
    ],
    'legend_align': 'left',
    'title': 'The 20 most dominant Taskmaster runs',
    'subtitle': 'Share of series points vs an even split (1.0 = an average contestant that series)',
    'source': 'Taskmaster UK series 1–21, contestant point totals',
    'filename': '02_top20_performances',
})

---
## Chart 3: Runaways vs nail-biters
Each series winner's lead over the runner-up, in share-of-points percentage points. Small bars = the series went down to the wire.

In [ ]:
render_chart({
    'type': 'single_ranked_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',
    'table': 'chart_margins',
    'category_col': 'label',
    'value_col': 'val',
    'label_col': 'val_label',
    'bar_color': '#005F73',
    'title': 'Taskmaster runaways and nail-biters',
    'subtitle': "Series winner's lead over the runner-up (share of points)",
    'source': 'Taskmaster UK series 1–21, contestant point totals',
    'filename': '03_winner_margins',
})

---
## Chart 4: Every contestant, ranked (oversized — NOT a standard social post)
The full 105-contestant ranking on the apples-to-apples metric, winner (teal) vs non-winner (red).

**⚠️ Sizing / where this can go.** This chart is exported at **1600×2600** (tall), NOT the `twitter_landscape` preset — 105 legible bars can't fit a 16:9 card.

- **OK to post / use:** website or blog embed, PDF/print, a link-out or dataset landing page, or on X/Twitter as an *attached image* (opens full-size when tapped) or a pinned ‘full ranking’ reply.
- **Do NOT use for:** in-feed social cards that crop to landscape (Twitter/LinkedIn timeline previews) or Instagram feed (needs 1080×1080 or 1080×1350) — it will be cropped and become unreadable.

For an in-feed post, use Chart 2 (top 20) instead, which carries the same story at 1600×900.

In [ ]:
# Rendered directly (not via render_chart) because it uses a CUSTOM height,
# not a platform preset. Saved to outputs/social/ with the _TALL suffix as a
# reminder that it must not be dropped into a landscape/square social card.
allr = conn.execute('SELECT * FROM chart_all_ranked').df()

img = single_ranked_bars(
    allr,
    category_col='label', value_col='val', total_label_col='val_label',
    color_col='bar_color',
    legend=[{'label': 'Won their series', 'color': C_TEAL},
            {'label': 'Did not win', 'color': C_RED}],
    legend_align='left',
    ref_line={'value': 1.0, 'label': 'average contestant (1.0)', 'color': '#EE9B00'},
    title='Every Taskmaster contestant, ranked',
    subtitle='Share of series points vs an even split (1.0 = an average contestant that series)',
    source='Taskmaster UK series 1–21, contestant point totals',
    label_font_size=13, total_font_size=12,
    img_width=1600, img_height=2600,
)

out_path = social_dir / '04_all_contestants_ranked_TALL.png'
img.save(out_path, format='PNG', optimize=True)
print(f'Saved chart -> {out_path}  (1600x2600 px, CUSTOM size — not a social preset)')
display(IPImage(filename=str(out_path)))

---
## Summary & cleanup

In [ ]:
conn.close()
pngs = sorted(social_dir.glob('*.png'))
print('=== ALL CHARTS COMPLETE ===')
print(f'\n✓ Generated {len(pngs)} publication-ready charts:')
for png in pngs:
    print(f'  • {png.name} ({png.stat().st_size/1024:.0f} KB)')
print('\nAll twitter_landscape (1600×900) with @unwelcomedata watermark.')